In [1]:
!pip install osmnx
!pip install NetworkX

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 1.2 MB/s eta 0:00:00


In [ ]:
import osmnx as ox
import matplotlib.pyplot as plt

# 1. Configuração do local
place = "Santa Cruz, Rio Grande do Norte, Brazil"

# 2. Download do grafo (malha viária de carros)
# Usamos simplify=True para remover nós desnecessários e deixar o mapa mais limpo
G = ox.graph_from_place(place, network_type="drive")

print(G)
print(f"Número de nós: {G.number_of_nodes()}")
print(f"Número de arestas: {G.number_of_edges()}")

# 3. Plotagem do mapa
# O comando ox.plot_graph retorna a 'fig' (figura) e o 'ax' (eixo) do Matplotlib
fig, ax = ox.plot_graph(
    G,
    node_size=0,          # Tamanho dos nós (0 esconde os pontos nos cruzamentos)
    edge_color="#2ecc71", # Cor das ruas (Verde esmeralda)
    edge_linewidth=0.5,   # Espessura das linhas
    bgcolor="#111111",    # Cor de fundo (Preto para contraste)
    show=False,           # False para podermos customizar antes de exibir
    close=False
)

# 4. Customização opcional (Título)
ax.set_title(f"Malha Viária: {place}")

# 5. Exibir o mapa na tela
plt.show()
print(G)

KeyboardInterrupt: 

In [6]:
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# =========================================================
# 1. DOWNLOAD DO GRAFO
# =========================================================

place = "Santa Cruz, Rio Grande do Norte, Brazil"

print(f"Baixando o grafo de {place}...")

G = ox.graph_from_place(place, network_type="drive")

print("\nInformações do grafo:")
print(G)

# =========================================================
# 2. CONVERSÃO PARA GRAFO SIMPLES
# =========================================================
# OSMnx retorna MultiDiGraph:
# - Multi = múltiplas ruas entre dois nós
# - DiGraph = ruas direcionadas
#
# Para métricas clássicas (k-core, hubs etc.)
# convertemos para Graph simples e não direcionado.

G_undirected = nx.Graph(G)

print("\nInformações do grafo convertido:")
print(G_undirected)

# =========================================================
# 3. GRAU DOS NÓS (HUBS)
# =========================================================

degree_dict = dict(G_undirected.degree())

top_hubs = sorted(degree_dict.items(), key=lambda x: x[1], reverse=True)[:10]

print("\n--- TOP 10 HUBS (MAIOR GRAU) ---")

for node, degree in top_hubs:
    print(f"Nó ID {node} | Grau: {degree}")
"""
# =========================================================
# 4. DISTRIBUIÇÃO DE GRAU
# =========================================================

degrees = list(degree_dict.values())

plt.figure(figsize=(8, 5))

plt.hist(
    degrees,
    bins=range(min(degrees), max(degrees) + 2),
    color='#2ecc71',
    edgecolor='black',
    align='left',
    alpha=0.8
)

plt.title(f"Distribuição de Grau - {place}")
plt.xlabel("Grau")
plt.ylabel("Frequência")

plt.xticks(range(min(degrees), max(degrees) + 1))

plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

"""

# =========================================================
# 5. CENTRALIDADES
# =========================================================

print("\nCalculando Betweenness Centrality...")

betweenness = nx.betweenness_centrality(
    G_undirected,
    normalized=True
)

print("Calculando Closeness Centrality...")

closeness = nx.closeness_centrality(G_undirected)

top_betweenness = sorted(
    betweenness.items(),
    key=lambda x: x[1],
    reverse=True
)[:3]

top_closeness = sorted(
    closeness.items(),
    key=lambda x: x[1],
    reverse=True
)[:3]

print("\n--- TOP 3 BETWEENNESS (Gargalos) ---")

for node, cent in top_betweenness:
    print(f"Nó ID {node} | Valor: {cent:.6f}")

print("\n--- TOP 3 CLOSENESS (Mais centrais) ---")

for node, cent in top_closeness:
    print(f"Nó ID {node} | Valor: {cent:.6f}")


#G_simple = nx.Graph(G)
print("\n Calculando K-core...")
core_number = nx.core_number(G_undirected)

max_core = max(core_number.values())
print(f"Maior core number: {max_core}")

# Nós pertencentes ao núcleo mais denso
main_core_nodes = [node for node, core in core_number.items() if core == max_core]
print(f"Número de nós no núcleo principal: {len(main_core_nodes)}")

# =========================================================
# 8. EXPORTAÇÃO PARA GEXF (GEPHI)
# =========================================================

print("\nPreparando exportação GEXF...")

# ---------- adiciona métricas ----------

nx.set_node_attributes(G_undirected, degree_dict, "degree")
nx.set_node_attributes(G_undirected, core_number, "core_number")
nx.set_node_attributes(G_undirected, betweenness, "betweenness")
nx.set_node_attributes(G_undirected, closeness, "closeness")

# =========================================================
# LIMPEZA DOS DADOS
# =========================================================

# -------- NÓS --------

for node, data in G_undirected.nodes(data=True):

    # latitude e longitude para o Gephi
    data["longitude"] = data.get("x")
    data["latitude"] = data.get("y")

    for key in list(data.keys()):

        value = data[key]

        # converte tipos incompatíveis
        if isinstance(value, (list, dict, set, tuple)):
            data[key] = str(value)

# -------- ARESTAS --------

for u, v, data in G_undirected.edges(data=True):

    for key in list(data.keys()):

        value = data[key]

        # REMOVE geometry
        if key == "geometry":
            del data[key]

        # osmid geralmente é lista
        elif key == "osmid":
            data[key] = str(value)

        # outros tipos incompatíveis
        elif isinstance(value, (list, dict, set, tuple)):
            data[key] = str(value)

# =========================================================
# EXPORTAÇÃO
# =========================================================

nome_arquivo = "santa_cruz_rede_urbana.gexf"

nx.write_gexf(G_undirected, nome_arquivo)

print(f"\nArquivo '{nome_arquivo}' exportado com sucesso!")
print("Abra no Gephi: File > Open > .gexf")

Baixando o grafo de Santa Cruz, Rio Grande do Norte, Brazil...

Informações do grafo:
MultiDiGraph with 1984 nodes and 5346 edges

Informações do grafo convertido:
Graph with 1984 nodes and 2783 edges

--- TOP 10 HUBS (MAIOR GRAU) ---
Nó ID 3964693509 | Grau: 6
Nó ID 3964715642 | Grau: 5
Nó ID 314993294 | Grau: 4
Nó ID 1828181337 | Grau: 4
Nó ID 2837621816 | Grau: 4
Nó ID 2837621823 | Grau: 4
Nó ID 2837621824 | Grau: 4
Nó ID 2837621832 | Grau: 4
Nó ID 2837621850 | Grau: 4
Nó ID 2837621855 | Grau: 4

Calculando Betweenness Centrality...
Calculando Closeness Centrality...

--- TOP 3 BETWEENNESS (Gargalos) ---
Nó ID 7428327102 | Valor: 0.153927
Nó ID 7410672473 | Valor: 0.144247
Nó ID 7413519201 | Valor: 0.135714

--- TOP 3 CLOSENESS (Mais centrais) ---
Nó ID 7428327102 | Valor: 0.049233
Nó ID 3964616875 | Valor: 0.049055
Nó ID 7429003086 | Valor: 0.048988

 Calculando K-core...
Maior core number: 2
Número de nós no núcleo principal: 1643

Preparando exportação GEXF...

Arquivo 'santa_cru